<a href="https://colab.research.google.com/github/abdiToldSo/Bitcamp-2025-Materials/blob/master/aiMLTrackCopy_of_AI_Workshop_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 ## Workshop: Basic Retrieval-Augmented Generation (RAG)

 This notebook demonstrates a simple RAG pipeline using Google Gemini for LLM responses and Pinecone as the vector database.

 ---

 ### Step 1: Setup and Install Dependencies
 Ensure you have the required libraries installed:
 ```bash
pip install sentence-transformers google-generativeai pinecone PyMuPDF
 ```


In [ ]:
!pip install google-generativeai pinecone PyMuPDF PyPDF2

In [ ]:
import os
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
import fitz
from google.generativeai import configure, GenerativeModel
import json


 ---
 ### Step 2: API Keys Setup
Configure the API keys needed to access Google Gemini and Pinecone services.
You'll need to replace these with your own API keys in a real application. We also initialize the
Pinecone client that we'll use to interact with our vector database. <br>
Gemini API KEY: https://aistudio.google.com/app/apikey <br>
Pinecone API KEY: https://www.pinecone.io/ (to create an account if you haven't already) -> API KEYS -> CREATE NEW API KEY


In [ ]:
#insert API keys, configure DB
gemini_api_key = ""
pinecone_api_key = ""
pinecone_env = "us-east-1"  # Example: 'us-east-1'

configure(api_key=gemini_api_key)
pc = Pinecone(
        api_key=pinecone_api_key,
  )

 ---

 ### Step 3: Initialize Vector Database
 We create (or connect to) an index in Pinecone to store and retrieve vector embeddings.

In [ ]:
index_name = "workshop" #name for your database

existing_indexes = [index["name"] for index in pc.list_indexes()]

# create only if it doesn't exist already
if index_name not in existing_indexes:
  pc.create_index(
      name=index_name,
      dimension= 384, # Replace with your model dimensions (this embedding model has 384 dimensions)
      metric= "cosine", # Replace with your model metric
      spec=ServerlessSpec(
          cloud="aws",
          region="us-east-1"
      )
  )

In [ ]:
index = pc.Index(index_name)
index

 ---

 ### Step 4: Embedding Model Setup
 We use OpenAI's `tall-MiniLM-L6-v2` model to embed documents and queries, which has 384 dimensions. This will very well be different from the embedding dimensions of the Gemini model, but we don't need to consider that since RAG is a standalone process that does not involve the LLM until the final context has been received

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_text(text):
    embeddings = model.encode(text)  # The model expects a list of texts
    return embeddings

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 ---

 ### Step 5: Ingest Documents into Pinecone
 Convert the extracted text from the PDF into embeddings and store them in Pinecone.
 These are helper methods to get your pdf embedded and stored in a vectorDB. Big pdfs can take a bit of time.

In [ ]:
from PyPDF2 import PdfReader

def extract_text_from_pdf(pdf_file_path):
    # Extract text from the entire PDF document
    pdf_text = ""
    with open(pdf_file_path, "rb") as file:
        reader = PdfReader(file)
        for page in reader.pages:
            pdf_text += page.extract_text()
    return pdf_text

def chunk_text(text, chunk_size=500):
    # Split text into smaller chunks based on the chunk_size (e.g., 500 characters)
    # This is a simple approach, there are ways to do better (having overlap, etc)
    chunks = []
    print("Total Chunks to be processed: ", len(text)//chunk_size)
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

def ingest_pdf(pdf_file_path, doc_id):
    print("Starting Text Extraction...")
    pdf_text = extract_text_from_pdf(pdf_file_path)
    print("Starting Chunking ...")
    text_chunks = chunk_text(pdf_text, chunk_size = 500 ) #mention chunk size
    print("Starting the Embedding process ...")
     # Generate embeddings for all text chunks at once
    embeddings = embed_text(text_chunks)
    print(embeddings.shape)
    print("Data prepared to upsert")
    # Prepare upsert payload
    upsert_data = [
        (f"{doc_id}_{i}", embedding.tolist(), {"text": chunk})
        for i, (embedding, chunk) in enumerate(zip(embeddings, text_chunks))
    ]
    # Upsert all at once
    index.upsert(upsert_data)
    print("data upserted :)")


In [ ]:
# Ingest an uploaded PDF document:
ingest_pdf("<your_pdf>", "doc1") #doc1 is a sample identifier, you can make it anything

Starting Text Extraction...
Starting Chunking ...
Total Chunks to be processed:  45
Starting the Embedding process ...
(46, 384)
Data prepared to upsert
data upserted :)


 ---

 ### Step 6: Retrieve Relevant Documents
 Given a query, retrieve the most relevant document using similarity search. We use all the functions already coded in the notebook to query the vector db

In [ ]:
def retrieve_relevant_docs(query, top_k=5):
    query_embedding = embed_text([query]) #the function expects a list of vectors
    query_embedding = query_embedding.tolist()  # Convert ndarray to list

    # Query Pinecone for the top_k most relevant chunks
    results = index.query(vector=query_embedding, top_k=top_k, include_metadata=True)

    # Return the relevant text chunks
    return [match["metadata"]["text"].replace("\n","") for match in results["matches"]]

In [ ]:
query = "<your query>" #user query
retrieved_docs = retrieve_relevant_docs(query)

In [ ]:
retrieved_docs

[' A grade available to everybody who makes an illegal offering to testudo right before finals even though we don’t have a final exam. By the way, this is not a joke, part of the course is courage, if you make a reflection about this we WILL give you a free A. It has happened before The following table: ,,,,"Grading Scale ",,,,,, "A+ ","97% ","B+ ","≥87.00% ","C+ ","',
 'ir progress on the project. The updates may be asked to be presented as write-ups, slideshows, or brief recorded videos. Details will be made available during the semester. [A/T]   [F: Formative (30%) - S: Summative (30%) - A: Applied (40%) /I: Individual (56%) - T: Team (44%)] Grades Your course grade is determined by your performance on the learning assessments in the co',
 'ted to post a main contribution, responses to other students, and a final contribution summarizing their take-aways from the exchange. [F/I]   Muddiest Point Clarifications (10% of final grade): Every other week, students will be asked to suggest


 ---

 ### Step 7: Generate Response using Google Gemini
 We use the retrieved context to generate a response with the LLM.

In [ ]:
gemini_model = GenerativeModel(model_name="gemini-2.5-pro-exp-03-25")

def generate_answer(query, context):
    prompt1 = f"Use ONLY the context provided to answer the query. Context: {context}\n\nQuestion: {query}\n\nAnswer:"
    response1 = gemini_model.generate_content(prompt1)
    prompt2 = f"answer this question {query}"
    response2=  gemini_model.generate_content(prompt2)
    return response1.text, response2.text

In [ ]:
answer1, answer2 = generate_answer(query, retrieved_docs)

 ---

 ### Step 8: Display Results
 We print the retrieved context and the final response.

In [ ]:
print("Query:", query)
print("\nRetrieved Context:", retrieved_docs[0])
print("\nRAG Answer:", answer1)

Query: Whats an easy way to get an A in the class

Retrieved Context:  A grade available to everybody who makes an illegal offering to testudo right before finals even though we don’t have a final exam. By the way, this is not a joke, part of the course is courage, if you make a reflection about this we WILL give you a free A. It has happened before The following table: ,,,,"Grading Scale ",,,,,, "A+ ","97% ","B+ ","≥87.00% ","C+ ","

RAG Answer: According to the provided text, an A grade is available to everybody who makes an illegal offering to Testudo right before finals (even though there isn't a final exam) and then makes a reflection about it. The text states, "if you make a reflection about this we WILL give you a free A."


In [ ]:
print("\nNormal Answer:", answer2)


Normal Answer: Unfortunately, I can't give you the specific grade distribution for INST123 because:

1.  **I don't know which institution you're referring to.** INST123 could be a course code at many different universities or colleges.
2.  **Grade distributions are specific institutional data.** This information isn't publicly available through a general AI like me. It's usually held internally by the university or college.
3.  **Distributions vary.** Even within the same institution, the grade distribution for INST123 can change significantly depending on the semester, the specific instructor, and the particular group of students enrolled.

**How to find this information:**

1.  **Specify the Institution:** Identify the university or college where INST123 is offered.
2.  **Check Official University Resources:**
    *   **Registrar's Office:** They might have official statistics, sometimes available upon request or through internal portals.
    *   **The Department Offering the Course

In [ ]:
index.delete(deleteAll=True)

{}